# Saving a Model and Using It in an Application

Training a model is only half the job. A model that only lives inside a notebook can't be used by anyone else. In this notebook we go step by step through:

1. **What** actually has to be saved (it is more than the model!)
2. **How** to save and load a model with each library
   - scikit-learn → `joblib`
   - XGBoost → its native JSON format
   - PyTorch → `state_dict` and checkpoints
3. **Metadata** – saving what you need to use the model correctly later
4. **Common pitfalls**
5. **Putting it into an application** – a reusable prediction module and a small web API

Libraries: `numpy`, `pandas`, `scikit-learn`, `xgboost`, `torch` (plus `joblib`, which comes with scikit-learn).

## 0. The big picture: what do we need to save?

A trained model is not just "the model object". To get the same predictions tomorrow, in another program, you need **all** of these:

| Artifact | Why it is needed |
|---|---|
| Model weights / parameters | The thing you learned |
| Preprocessing (scaler, encoder, ...) | New data must be transformed *exactly* like the training data |
| Model architecture / config | Needed to rebuild a PyTorch model before loading weights |
| Feature names and order | Columns in the wrong order give silently wrong predictions |
| Metadata (versions, metrics, date) | So you can reproduce, debug and compare later |

The rule of thumb: **save everything that `predict()` depends on.**

## 1. Setup

In [ ]:
# Only run this if a package is missing (xgboost is not part of a default install)
%pip install numpy pandas scikit-learn xgboost torch

In [ ]:
import json
import platform
from datetime import datetime
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import xgboost as xgb
import sklearn
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Everything we save goes in one folder
ARTIFACT_DIR = Path("saved_models")
ARTIFACT_DIR.mkdir(exist_ok=True)

SEED = 42
rng = np.random.default_rng(SEED)
torch.manual_seed(SEED)

print("Artifacts will be saved to:", ARTIFACT_DIR.resolve())

## 2. Data: a small customer-churn dataset

To keep this notebook self-contained we generate synthetic data with numpy and pandas. The task: predict whether a customer **churns** (cancels) from a few numeric features and one categorical feature (`plan`). Having a categorical column matters, because it forces us to save an encoder too.

In [ ]:
n = 3000
df = pd.DataFrame({
    "age": rng.integers(18, 75, n),
    "income": rng.normal(50_000, 15_000, n).round(2),
    "tenure_months": rng.integers(1, 72, n),
    "support_tickets": rng.poisson(2, n),
    "plan": rng.choice(["basic", "standard", "premium"], n, p=[0.5, 0.3, 0.2]),
})

# Hidden rule that generates the label (the model has to discover it)
logit = (
    -1.0
    + 0.45 * df["support_tickets"]
    - 0.04 * df["tenure_months"]
    - 0.00002 * (df["income"] - 50_000)
    + df["plan"].map({"basic": 0.8, "standard": 0.0, "premium": -0.8})
    + rng.normal(0, 0.5, n)
)
df["churned"] = (rng.random(n) < 1 / (1 + np.exp(-logit))).astype(int)

df.head()

In [ ]:
NUMERIC_FEATURES = ["age", "income", "tenure_months", "support_tickets"]
CATEGORICAL_FEATURES = ["plan"]
FEATURES = NUMERIC_FEATURES + CATEGORICAL_FEATURES  # the order matters!
TARGET = "churned"

X_train, X_test, y_train, y_test = train_test_split(
    df[FEATURES], df[TARGET], test_size=0.2, random_state=SEED, stratify=df[TARGET]
)
print(X_train.shape, X_test.shape, f"churn rate: {y_train.mean():.2%}")

## 3. scikit-learn: save a whole `Pipeline` with `joblib`

### The idea
Instead of saving "a scaler" and "a model" separately and hoping you reuse them in the right order, put them in a **`Pipeline`**. Then you save **one object** and `pipeline.predict(raw_dataframe)` does the preprocessing and the prediction.

### Step 1 – build and train

In [ ]:
preprocessor = ColumnTransformer([
    ("num", StandardScaler(), NUMERIC_FEATURES),
    ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), CATEGORICAL_FEATURES),
])

sk_pipeline = Pipeline([
    ("prep", preprocessor),
    ("model", RandomForestClassifier(n_estimators=200, max_depth=6, random_state=SEED)),
])

sk_pipeline.fit(X_train, y_train)

sk_proba = sk_pipeline.predict_proba(X_test)[:, 1]
print(f"sklearn  accuracy: {accuracy_score(y_test, sk_proba > 0.5):.3f}  AUC: {roc_auc_score(y_test, sk_proba):.3f}")

### Step 2 – save
`joblib.dump` is the standard way to save scikit-learn objects. It is built on `pickle` but is more efficient for objects that hold large numpy arrays (like a forest).

In [ ]:
SK_PATH = ARTIFACT_DIR / "sklearn_pipeline.joblib"
joblib.dump(sk_pipeline, SK_PATH)
print(f"Saved {SK_PATH}  ({SK_PATH.stat().st_size / 1024:.0f} KB)")

### Step 3 – load and verify
**Always verify** that the loaded model gives identical predictions. This is the cheapest test you can write and it catches most saving bugs.

In [ ]:
loaded_sk = joblib.load(SK_PATH)

loaded_proba = loaded_sk.predict_proba(X_test)[:, 1]
assert np.allclose(sk_proba, loaded_proba), "Loaded model gives different predictions!"
print("Loaded sklearn pipeline gives identical predictions ✔")

# It works on raw data - the pipeline does the scaling and encoding itself
new_customer = pd.DataFrame([{"age": 34, "income": 42_000, "tenure_months": 3,
                              "support_tickets": 6, "plan": "basic"}])
print("Churn probability:", loaded_sk.predict_proba(new_customer)[0, 1].round(3))

> ⚠️ **Security:** `joblib`/`pickle` files can execute arbitrary code when loaded. **Only load files you created yourself or fully trust.** Never load a `.joblib` or `.pkl` from an unknown source.

## 4. XGBoost: use the native format

XGBoost models *can* be pickled, but the XGBoost docs recommend its own format: it is stable across XGBoost versions, language-independent (R, C++, Java can read it) and safe to load. Use the **`.json`** extension (or `.ubj` for a smaller binary version).

### An important catch
`save_model` stores **only the trees** – no scaler, no encoder. So we must save the preprocessor ourselves. Here we reuse the preprocessor that was fitted inside the sklearn pipeline.

In [ ]:
# Reuse the already-fitted preprocessor (do NOT refit it, or the features would change)
fitted_prep = sk_pipeline.named_steps["prep"]
X_train_t = fitted_prep.transform(X_train)
X_test_t = fitted_prep.transform(X_test)

xgb_model = xgb.XGBClassifier(
    n_estimators=200, max_depth=3, learning_rate=0.1,
    eval_metric="logloss", random_state=SEED,
)
xgb_model.fit(X_train_t, y_train)

xgb_proba = xgb_model.predict_proba(X_test_t)[:, 1]
print(f"XGBoost  accuracy: {accuracy_score(y_test, xgb_proba > 0.5):.3f}  AUC: {roc_auc_score(y_test, xgb_proba):.3f}")

### Save the model and the preprocessor

In [ ]:
XGB_PATH = ARTIFACT_DIR / "xgboost_model.json"
PREP_PATH = ARTIFACT_DIR / "preprocessor.joblib"

xgb_model.save_model(XGB_PATH)       # trees -> JSON
joblib.dump(fitted_prep, PREP_PATH)  # scaler + encoder -> joblib

print("Saved:", XGB_PATH.name, "and", PREP_PATH.name)

### Load and verify
Create an empty `XGBClassifier` and call `load_model`. You don't need to repeat the hyperparameters – they are stored in the file.

In [ ]:
loaded_xgb = xgb.XGBClassifier()
loaded_xgb.load_model(XGB_PATH)
loaded_prep = joblib.load(PREP_PATH)

loaded_xgb_proba = loaded_xgb.predict_proba(loaded_prep.transform(X_test))[:, 1]
assert np.allclose(xgb_proba, loaded_xgb_proba, atol=1e-6)
print("Loaded XGBoost model gives identical predictions ✔")

## 5. PyTorch: save the `state_dict`, not the whole model

PyTorch gives you two options:

| Option | How | Verdict |
|---|---|---|
| Save the whole model | `torch.save(model, path)` | ❌ Pickles the class by *name and file location*. Breaks as soon as you rename or move the code. |
| Save the `state_dict` | `torch.save(model.state_dict(), path)` | ✅ Saves only the tensors (weights). **Recommended.** |

With a `state_dict` you must recreate the architecture in code and then load the weights into it. So the **class definition has to live in a normal `.py` file** that both your training notebook *and* your application can import. We write it to a file now and import it back.

In [ ]:
APP_DIR = Path("app")
APP_DIR.mkdir(exist_ok=True)

In [ ]:
%%writefile app/model_definition.py
"""The PyTorch architecture. Shared by training (notebook) and serving (application)."""
import torch.nn as nn


class ChurnMLP(nn.Module):
    def __init__(self, in_features: int, hidden: int = 32, dropout: float = 0.2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_features, hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, 1),  # raw logit; sigmoid is applied at prediction time
        )

    def forward(self, x):
        return self.net(x).squeeze(-1)

### Step 1 – train

In [ ]:
import sys
sys.path.insert(0, str(APP_DIR.resolve()))
from model_definition import ChurnMLP

TORCH_CONFIG = {"in_features": X_train_t.shape[1], "hidden": 32, "dropout": 0.2}
model = ChurnMLP(**TORCH_CONFIG)

loss_fn = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-2)

X_tr = torch.tensor(X_train_t, dtype=torch.float32)
y_tr = torch.tensor(y_train.to_numpy(), dtype=torch.float32)
X_te = torch.tensor(X_test_t, dtype=torch.float32)

loader = torch.utils.data.DataLoader(
    torch.utils.data.TensorDataset(X_tr, y_tr), batch_size=64, shuffle=True
)

EPOCHS = 30
for epoch in range(1, EPOCHS + 1):
    model.train()
    for xb, yb in loader:
        optimizer.zero_grad()
        loss = loss_fn(model(xb), yb)
        loss.backward()
        optimizer.step()

model.eval()
with torch.no_grad():
    torch_proba = torch.sigmoid(model(X_te)).numpy()
print(f"PyTorch  accuracy: {accuracy_score(y_test, torch_proba > 0.5):.3f}  AUC: {roc_auc_score(y_test, torch_proba):.3f}")

### Step 2 – save (weights only)
We also save the **config** (`in_features`, `hidden`, ...) as JSON, because we need it to rebuild the model before loading the weights.

In [ ]:
TORCH_PATH = ARTIFACT_DIR / "torch_model.pt"
TORCH_CONFIG_PATH = ARTIFACT_DIR / "torch_config.json"

torch.save(model.state_dict(), TORCH_PATH)
TORCH_CONFIG_PATH.write_text(json.dumps(TORCH_CONFIG, indent=2))

print("state_dict keys:", list(model.state_dict().keys()))

### Step 3 – load and verify
Three things people forget:
1. **Rebuild the model first**, with the same config.
2. **`model.eval()`** – otherwise dropout stays active and predictions are random.
3. **`map_location="cpu"`** – lets you load a GPU-trained model on a machine without a GPU. `weights_only=True` makes loading safer (only tensors are allowed, no arbitrary code).

In [ ]:
config = json.loads(TORCH_CONFIG_PATH.read_text())
loaded_torch = ChurnMLP(**config)
loaded_torch.load_state_dict(torch.load(TORCH_PATH, map_location="cpu", weights_only=True))
loaded_torch.eval()

with torch.no_grad():
    loaded_torch_proba = torch.sigmoid(loaded_torch(X_te)).numpy()

assert np.allclose(torch_proba, loaded_torch_proba, atol=1e-6)
print("Loaded PyTorch model gives identical predictions ✔")

### Bonus – a training checkpoint (to resume training)
For inference the weights are enough. If you want to **continue training later** (or recover from a crash) also save the optimizer state and the epoch. Adam keeps running averages, so without its state a resumed run behaves differently.

In [ ]:
CHECKPOINT_PATH = ARTIFACT_DIR / "torch_checkpoint.pt"

torch.save({
    "epoch": EPOCHS,
    "model_state": model.state_dict(),
    "optimizer_state": optimizer.state_dict(),
    "config": TORCH_CONFIG,
}, CHECKPOINT_PATH)

# --- later: resume ---
ckpt = torch.load(CHECKPOINT_PATH, map_location="cpu", weights_only=True)
resumed = ChurnMLP(**ckpt["config"])
resumed.load_state_dict(ckpt["model_state"])
resumed_opt = torch.optim.Adam(resumed.parameters(), lr=1e-2)
resumed_opt.load_state_dict(ckpt["optimizer_state"])
print(f"Checkpoint restored, would continue from epoch {ckpt['epoch'] + 1}")

## 6. Save metadata
Six months from now you will want to know *which* data, *which* library versions and *how good* each model was. A small `metadata.json` next to the models costs nothing. It also stores the **feature list in the correct order**, which the application uses to validate input.

In [ ]:
metadata = {
    "created": datetime.now().isoformat(timespec="seconds"),
    "features": FEATURES,
    "numeric_features": NUMERIC_FEATURES,
    "categorical_features": CATEGORICAL_FEATURES,
    "allowed_plans": sorted(df["plan"].unique().tolist()),
    "target": TARGET,
    "threshold": 0.5,
    "metrics_auc": {
        "sklearn": round(roc_auc_score(y_test, sk_proba), 4),
        "xgboost": round(roc_auc_score(y_test, xgb_proba), 4),
        "torch": round(roc_auc_score(y_test, torch_proba), 4),
    },
    "versions": {
        "python": platform.python_version(),
        "numpy": np.__version__,
        "pandas": pd.__version__,
        "scikit-learn": sklearn.__version__,
        "xgboost": xgb.__version__,
        "torch": torch.__version__,
    },
}

(ARTIFACT_DIR / "metadata.json").write_text(json.dumps(metadata, indent=2))
print(json.dumps(metadata, indent=2))

Our artifact folder now looks like this:

In [ ]:
for p in sorted(ARTIFACT_DIR.iterdir()):
    print(f"{p.name:28} {p.stat().st_size / 1024:8.1f} KB")

## 7. Common pitfalls (cheat sheet)

| Pitfall | What goes wrong | Fix |
|---|---|---|
| Saving only the model, not the scaler/encoder | Predictions are garbage on new data | Use a `Pipeline`, or save the preprocessor too |
| Refitting the scaler at prediction time | Different scaling than training | Only `.transform()` after loading, never `.fit()` |
| Forgetting `model.eval()` in PyTorch | Dropout/BatchNorm still in training mode → noisy predictions | Always call `eval()` and use `torch.no_grad()` |
| `torch.save(model)` of the whole object | Breaks when code is renamed/moved | Save the `state_dict` + config |
| Class definition only inside the notebook | Application can't rebuild the model | Put it in a `.py` module |
| Different column order | Silent wrong predictions | Reorder with the saved feature list |
| Loading with a different library version | Warnings, errors or changed behaviour | Pin versions (`requirements.txt`), record them in metadata |
| Loading untrusted pickle/joblib/`torch.load` | Arbitrary code execution | Only load trusted files; `weights_only=True` for torch |
| Never testing the loaded model | Bugs found in production | Assert loaded predictions == original predictions |

## 8. Using the saved models in an application

Now the second half of the question. An application must **never train** – it only **loads** the saved artifacts and predicts. The standard structure is:

```
my_project/
├── train.ipynb / train.py      <- produces the artifacts (what we just did)
├── saved_models/               <- the artifacts
│   ├── sklearn_pipeline.joblib
│   ├── preprocessor.joblib
│   ├── xgboost_model.json
│   ├── torch_model.pt / torch_config.json
│   └── metadata.json
└── app/
    ├── model_definition.py     <- PyTorch class (shared)
    ├── model_service.py        <- loads models, validates input, predicts
    └── api.py                  <- web API that calls model_service
```

The key design rules:
- **Load once at startup**, not on every request (loading is slow, predicting is fast).
- **Put all model logic in one class/module** (`model_service.py`) so the web framework, a CLI or a batch job can all reuse it.
- **Validate input** before it reaches the model.

### Step 1 – a prediction service

In [ ]:
%%writefile app/model_service.py
"""Loads all saved models once and exposes a single `predict` method."""
import json
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import torch
import xgboost as xgb

from model_definition import ChurnMLP

DEFAULT_ARTIFACT_DIR = Path(__file__).resolve().parent.parent / "saved_models"


class ChurnPredictor:
    def __init__(self, artifact_dir: Path = DEFAULT_ARTIFACT_DIR):
        artifact_dir = Path(artifact_dir)

        self.metadata = json.loads((artifact_dir / "metadata.json").read_text())
        self.features = self.metadata["features"]
        self.threshold = self.metadata["threshold"]

        # scikit-learn: pipeline includes its own preprocessing
        self.sk_pipeline = joblib.load(artifact_dir / "sklearn_pipeline.joblib")

        # XGBoost + PyTorch work on the preprocessed array
        self.preprocessor = joblib.load(artifact_dir / "preprocessor.joblib")

        self.xgb_model = xgb.XGBClassifier()
        self.xgb_model.load_model(artifact_dir / "xgboost_model.json")

        config = json.loads((artifact_dir / "torch_config.json").read_text())
        self.torch_model = ChurnMLP(**config)
        self.torch_model.load_state_dict(
            torch.load(artifact_dir / "torch_model.pt", map_location="cpu", weights_only=True)
        )
        self.torch_model.eval()  # important!

    def _validate(self, records: list[dict]) -> pd.DataFrame:
        df = pd.DataFrame(records)
        missing = [c for c in self.features if c not in df.columns]
        if missing:
            raise ValueError(f"Missing features: {missing}")
        bad_plans = set(df["plan"]) - set(self.metadata["allowed_plans"])
        if bad_plans:
            raise ValueError(f"Unknown plan(s): {sorted(bad_plans)}")
        return df[self.features]  # enforce the training column order

    def predict(self, records: list[dict]) -> list[dict]:
        """records: list of raw feature dicts -> list of probabilities per model."""
        df = self._validate(records)

        p_sk = self.sk_pipeline.predict_proba(df)[:, 1]

        X = self.preprocessor.transform(df)
        p_xgb = self.xgb_model.predict_proba(X)[:, 1]

        with torch.no_grad():
            p_torch = torch.sigmoid(self.torch_model(torch.tensor(X, dtype=torch.float32))).numpy()

        p_avg = np.mean([p_sk, p_xgb, p_torch], axis=0)

        return [
            {
                "sklearn": float(a), "xgboost": float(b), "torch": float(c),
                "churn_probability": float(d),
                "will_churn": bool(d >= self.threshold),
            }
            for a, b, c, d in zip(p_sk, p_xgb, p_torch, p_avg)
        ]

### Step 2 – try the service
This is exactly what a real application does: import the class, create **one** instance, call `predict`. No training code, no dataset – just the saved artifacts.

In [ ]:
import importlib
import model_service
importlib.reload(model_service)  # pick up edits if you re-run the cell above

predictor = model_service.ChurnPredictor(ARTIFACT_DIR)

customers = [
    {"age": 34, "income": 42_000, "tenure_months": 3,  "support_tickets": 6, "plan": "basic"},
    {"age": 58, "income": 80_000, "tenure_months": 60, "support_tickets": 0, "plan": "premium"},
]
pd.DataFrame(predictor.predict(customers), index=["new, many tickets", "loyal premium"]).round(3)

The first customer (short tenure, many support tickets, basic plan) gets a high churn probability, the loyal premium customer a low one – all three models agree, and the average is returned as the final answer.

Bad input is rejected with a clear error instead of a silent wrong answer:

In [ ]:
try:
    predictor.predict([{"age": 30, "income": 50_000, "tenure_months": 5, "support_tickets": 1, "plan": "gold"}])
except ValueError as e:
    print("Rejected:", e)

### Step 3 – expose it as a web API (FastAPI)
The service class doesn't know anything about HTTP, so wrapping it is short. Install once with `pip install fastapi uvicorn`.

Note the model is loaded **once at import time** – every request reuses it.

In [ ]:
%%writefile app/api.py
"""Run from the project root with:  uvicorn app.api:app --reload  (cd into app/ first, see notebook)"""
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field

from model_service import ChurnPredictor

app = FastAPI(title="Churn prediction API")
predictor = ChurnPredictor()  # loaded once at startup


class Customer(BaseModel):
    age: int = Field(ge=18, le=120)
    income: float = Field(ge=0)
    tenure_months: int = Field(ge=0)
    support_tickets: int = Field(ge=0)
    plan: str


@app.get("/health")
def health():
    return {"status": "ok", "model_created": predictor.metadata["created"]}


@app.post("/predict")
def predict(customer: Customer):
    try:
        return predictor.predict([customer.model_dump()])[0]
    except ValueError as e:
        raise HTTPException(status_code=422, detail=str(e))

Start the server from a terminal:

```bash
cd app
uvicorn api:app --reload
```

and call it (or open `http://127.0.0.1:8000/docs` for an interactive page):

```bash
curl -X POST http://127.0.0.1:8000/predict \
  -H "Content-Type: application/json" \
  -d '{"age": 34, "income": 42000, "tenure_months": 3, "support_tickets": 6, "plan": "basic"}'
```

The same `ChurnPredictor` could just as well be used in a **Streamlit/Gradio dashboard**, a **batch script** that scores a CSV, or a **scheduled job** – that is the benefit of keeping model logic separate from the interface.

### Step 4 – batch scoring with pandas
Not every application is a server. Often you just score a file of customers:

In [ ]:
batch = X_test.head(5).to_dict(orient="records")
scored = pd.concat([X_test.head(5).reset_index(drop=True), pd.DataFrame(predictor.predict(batch))], axis=1)
scored.round(3)

## 9. Checklist for going to production

1. ✅ Save **everything** `predict` needs: model, preprocessing, config, feature order.
2. ✅ Use the library's recommended format (`joblib` for sklearn, `.json` for XGBoost, `state_dict` for PyTorch).
3. ✅ Verify the loaded model reproduces the original predictions.
4. ✅ Keep the model class in a `.py` file, not just in the notebook.
5. ✅ Record library versions and pin them in `requirements.txt` (`pip freeze > requirements.txt`).
6. ✅ Load models **once** at startup; validate input on every request.
7. ✅ Never load model files from untrusted sources.
8. ✅ Version your artifacts (e.g. `saved_models/v1/`, `v2/`) so you can roll back.

### Other options worth knowing
- **ONNX** (`torch.onnx.export`, `skl2onnx`): a portable format that runs without Python or even in other languages. Good for deployment to non-Python systems.
- **TorchScript** (`torch.jit.script`/`trace`): serialises PyTorch code *and* weights, so the class definition isn't needed at load time.
- **MLflow**: a model registry that tracks versions, metrics and artifacts for you once you have many models.